In [8]:
import pandas as pd
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os 
pio.renderers.default = "notebook" 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, accuracy_score, roc_curve, auc

In [2]:
#configuracion de carpetas 
CARPETA_SALIDA = "Graficos_Informe"

#Creamos la carpeta automáticamente
if not os.path.exists(CARPETA_SALIDA):
    os.makedirs(CARPETA_SALIDA)
    print(f"Carpeta creada: {CARPETA_SALIDA}")
else:
    print(f"La carpeta '{CARPETA_SALIDA}' ya existe. Los gráficos se guardarán ahí.")

#paleta
COLORES = {
    'Verde_Fuerte': '#74b404', 
    'Verde_Claro':  '#cdfc7d', 
    'Rojo_Corp':    '#aa044c', 
    'Morado_Os':    '#876784',
    'Morado_Cl':    '#b09eae'
}

#Creamos funcion para guardar aqui los graficos
def guardar_grafico(fig, nombre_archivo):
    """
    Guarda el gráfico en HTML (interactivo) y PNG (estático) 
    dentro de la carpeta organizada automáticamente.
    """
    # Guardar HTML (Interactivo)
    ruta_html = os.path.join(CARPETA_SALIDA, f"{nombre_archivo}.html")
    fig.write_html(ruta_html)
    
    print(f"Gráfico guardado: {ruta_html}")

#Carga de datos
def cargar_y_preparar_datos(ruta_archivo):
    print(f"Cargando datos desde: {ruta_archivo}")
    try:
        df = pd.read_excel(ruta_archivo)
        
        df_viv = df[df['Proposito'].astype(str).str.contains('Vivienda', case=False, na=False)].copy()
        df_viv['Impago_Label'] = df_viv['Impago'].map({0: 'Pagado', 1: 'Impago'})
        
        if 'Posesion_Hipoteca' in df_viv.columns:
            df_viv['Tiene_Hipoteca'] = df_viv['Posesion_Hipoteca'].map({0: 'No tiene', 1: 'Sí tiene'})
            
        col_fiador = 'Fiador' if 'Fiador' in df_viv.columns else 'Cofirmante'
        df_viv['Fiador_Label'] = df_viv[col_fiador].map({0: 'Sin Fiador', 1: 'Con Fiador'})
            
        return df_viv
    except Exception as e:
        print(f"Error: {e}")
        return None

#Cargamos
ruta_real = os.path.join('..', 'Datos', 'Originales', 'información_préstamos.xlsx')
df_viv = cargar_y_preparar_datos(ruta_real)

Carpeta creada: Graficos_Informe
Cargando datos desde: ..\Datos\Originales\información_préstamos.xlsx


**REGRESION LOGISTICA**

In [ ]:
# Seleccionamos las variables que SÍ nos sirven para predecir
vars_numericas = ['Ingresos', 'Monto_Inicial', 'Edad', 'Scoring_Crediticio', 
                  'Meses_Empleo', 'Num_Creditos', 'Ratio_Deuda_Ingresos', 
                  'Personas_Cargo', 'Ratio_Interes', 'Duracion']

vars_texto = ['Estado_Civil', 'Estudios', 'Tipo_Jornada_Laboral', 'Fiador_Label']

# Preparamos el dataset para el modelo
df_model = df_viv[vars_numericas + vars_texto + ['Impago']].copy()

# Convertimos texto a números (One-Hot Encoding)
df_model = pd.get_dummies(df_model, columns=vars_texto, drop_first=True)

# Separamos Predictores (X) y Objetivo (y)
X = df_model.drop('Impago', axis=1)
y = df_model['Impago']

# División: 70% para entrenar, 30% para examinar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#2. ENTRENAMIENTO DEL MODELO (REGRESIÓN LOGÍSTICA)
# max_iter=1000 para asegurar que las matemáticas converjan bien
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train, y_train)

#3. PREDICCIONES
y_pred = modelo.predict(X_test)         # Predicción final (0 o 1)
y_prob = modelo.predict_proba(X_test)[:, 1] # Probabilidad de riesgo (0% a 100%)

#4. REPORTE DE MÉTRICAS (TEXTO)
print(f"Exactitud (Accuracy): {accuracy_score(y_test, y_pred):.2%}")
print("-" * 30)
print("Informe Detallado:")
print(classification_report(y_test, y_pred, target_names=['Pagado', 'Impago']))

Exactitud Global (Accuracy): 90.08%
------------------------------
Informe Detallado:
              precision    recall  f1-score   support

      Pagado       0.90      1.00      0.95     13839
      Impago       0.66      0.03      0.05      1547

    accuracy                           0.90     15386
   macro avg       0.78      0.51      0.50     15386
weighted avg       0.88      0.90      0.86     15386



c:\Users\alari\.conda\envs\RETO_07_MORADO\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning:

lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



In [11]:
# --- GRÁFICO 13: MATRIZ DE CONFUSIÓN (VERDAD vs PREDICCIÓN) ---
cm = confusion_matrix(y_test, y_pred)

fig13 = px.imshow(
    cm,
    text_auto=True,
    x=['Pred: Pagará', 'Pred: Impago'],
    y=['Realidad: Pagó', 'Realidad: Impagó'],
    color_continuous_scale=[[0, COLORES['Verde_Fuerte']], [1, COLORES['Rojo_Corp']]],
    title="<b>13. Evaluación: Matriz de Confusión</b><br><sup>¿Cuántos morosos se nos han escapado? (Falsos Negativos)</sup>"
)
fig13.update_layout(template="plotly_white", coloraxis_showscale=False)
fig13.show()
#guardar_grafico(fig13, "13_Matriz_Confusion")


# --- GRÁFICO 14: CURVA ROC (CAPACIDAD DE DISCRIMINACIÓN) ---
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig14 = px.area(
    x=fpr, y=tpr,
    title=f"<b>14. Curva ROC (AUC = {roc_auc:.2f})</b><br><sup>Cuanto más curva hacia arriba, mejor modelo</sup>",
    labels=dict(x='Tasa de Falsos Positivos (Ruido)', y='Tasa de Verdaderos Positivos (Acierto)'),
)
fig14.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1) # Línea base
fig14.update_traces(line_color=COLORES['Rojo_Corp'], fillcolor='rgba(170, 4, 76, 0.2)') # Relleno rojo suave
fig14.update_layout(template="plotly_white", xaxis_range=[0, 1], yaxis_range=[0, 1])
fig14.show()
#guardar_grafico(fig14, "14_Curva_ROC")


#IMPORTANCIA DE VARIABLES
# Extraemos los "pesos" que el modelo ha dado a cada variable
coeficientes = pd.DataFrame({
    'Variable': X.columns,
    'Peso': modelo.coef_[0]
}).sort_values(by='Peso', ascending=True) # Ordenamos para ver el ranking

# Colores: Rojo si aumenta el riesgo (Peso > 0), Verde si lo baja (Peso < 0)
coeficientes['Color'] = coeficientes['Peso'].apply(lambda x: COLORES['Rojo_Corp'] if x > 0 else COLORES['Verde_Fuerte'])

fig15 = px.bar(
    coeficientes, 
    x='Peso', 
    y='Variable', 
    orientation='h',
    title="<b>15. ¿Qué dispara el Riesgo? (Importancia de Variables)</b><br><sup>Derecha (Rojo): Aumenta Probabilidad de Impago | Izquierda (Verde): Protege contra el Impago</sup>",
    color='Color',
    color_discrete_map="identity" # Usamos los colores definidos en la columna 'Color'
)
fig15.update_layout(template="plotly_white", height=800)
fig15.show()
#guardar_grafico(fig15, "15_Importancia_Variables")